# Análise da segurança pública de Joinville/SC

Notebook acadêmico — SENAI/LAB365, Visualização de Dados e BI.

**Pergunta:** como evoluem os indicadores municipais publicados e quais comparações são sustentadas pelas fontes?

O roteiro preserva a análise Sinesp/MJSP (seções 1–13) e apresenta uma investigação **separada** de violência doméstica SSP/SC (seção 14).
Sinesp: 911 observações, 12 séries, janeiro/2020–setembro/2026 (PRF até agosto). SSP/SC: 216 células, 9 categorias, 2023–2024.
Vítimas, mandados e “Fato Comunicado” são unidades distintas; não há total geral nem união definitiva das fontes.

**Como estudar:** escolha o kernel da `.venv` e execute todas as células em ordem. Confira as tabelas antes de interpretar as figuras.
O [README](../README.md) reúne fontes, obtenção, versões, regras e pendências. O [ETL](../scripts/etl_sinesp.py) reproduz os sete XLSX;
aqui a leitura bruta de 2026 ilustra o tratamento e é conferida contra a edição aprovada pelo hash.
Saídas são geradas no kernel; o notebook versionado permanece sem outputs volumosos.


## 1. Fonte oficial

- **Sinesp VDE / MJSP** — base nacional dos gestores estaduais, um arquivo XLSX por ano (`bancovde-2020.xlsx` a `bancovde-2026.xlsx`).
- Recorte: UF `SC`, município `JOINVILLE`; abrangências Estadual, Polícia Federal (mandados, desde 01/2025) e Polícia Rodoviária Federal.
- Período disponível: **janeiro/2020 a setembro/2026** (a série da PRF termina em agosto/2026). 2026 é um ano parcial.
- Os XLSX ficam em `dados/brutos/sinesp/` (fora do Git). A base aprovada usada pelo painel está em `dashboard/data_publica/`.

## 2. Bibliotecas

- `pandas` organiza os dados em tabelas e calcula somas, médias e agrupamentos.
- Os módulos `dashboard.*` são do projeto: carregam a base aprovada e fazem os cálculos e gráficos já validados.

In [ ]:
import sys
from pathlib import Path

# O notebook roda dentro da pasta notebooks/; para importar os módulos do projeto, a pasta-raiz entra no caminho do Python.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

from dashboard import calculos, graficos
from dashboard import dados_publicos as dp
from src.pipeline import atualizacao_dados as au

pd.set_option("display.max_columns", 20)
print(f"Pandas {pd.__version__} | pasta do projeto: {RAIZ}")

# Renderização das figuras no kernel Jupyter.
%matplotlib inline

## 3. Diretórios

- `dados/brutos/sinesp/`: XLSX oficiais (originais, sem alteração).
- `dados/processados/sinesp/`: saídas locais do ETL (`scripts/etl_sinesp.py`).
- `dashboard/data_publica/`: base aprovada e publicada (2 CSVs + manifesto com SHA-256).

In [ ]:
PASTA_BRUTOS = RAIZ / "dados" / "brutos" / "sinesp"
PASTA_PROCESSADOS = RAIZ / "dados" / "processados" / "sinesp"

# rglob procura os XLSX também nas subpastas (cada edição oficial fica em uma pasta identificada).
arquivos_xlsx = sorted(PASTA_BRUTOS.rglob("bancovde-*.xlsx")) if PASTA_BRUTOS.is_dir() else []
print(f"XLSX encontrados: {len(arquivos_xlsx)}")
print(f"Base publicada: {dp.DIRETORIO_DADOS}")

## 4. Leitura dos arquivos

O painel e a análise usam a **base aprovada** (911 observações mensais). Ela é lida com `pd.read_csv()` pela função `dp.carregar_sinesp()`, que também confere colunas, tipos e regras de qualidade.

Se o XLSX oficial de 2026 estiver disponível, também mostramos como ele é lido (apenas as linhas de Joinville). Sem os XLSX, o notebook segue em **modo demonstração**, com a base aprovada — isso não reproduz a extração (que está no ETL).

In [ ]:
# Lê a base aprovada (CSV publicado) e mostra as cinco primeiras linhas.
dados_sinesp = dp.carregar_sinesp()
print(f"Base aprovada: {dados_sinesp.shape[0]} observações × {dados_sinesp.shape[1]} colunas")
dados_sinesp.head()

In [ ]:
# Procura a edição oficial de 2026 com o mesmo SHA-256 registrado no manifesto da base aprovada.
manifesto = dp.carregar_manifesto()
hash_2026 = manifesto["origem_dos_dados"]["sinesp_xlsx_sha256"]["2026"]
xlsx_2026 = [c for c in arquivos_xlsx if c.name == "bancovde-2026.xlsx"]

dados_brutos_2026 = None
for caminho in xlsx_2026:
    import hashlib
    if hashlib.sha256(caminho.read_bytes()).hexdigest() == hash_2026:
        # ler_sinesp_xlsx percorre o XML da planilha e devolve só as linhas de SC/JOINVILLE (14 colunas originais).
        dados_brutos_2026 = au.ler_sinesp_xlsx(caminho)
        break

if dados_brutos_2026 is None:
    print("MODO DEMONSTRAÇÃO: XLSX oficial de 2026 não encontrado; a análise segue com a base aprovada (dado derivado, não bruto).")
else:
    print(f"XLSX 2026 lido: {len(dados_brutos_2026)} linhas de Joinville")

## 5. Inspeção bruta

Quando o XLSX está disponível, olhamos as linhas como vieram da fonte: a data de referência é um número (serial do Excel), `total_vitima` traz as vítimas e `total` traz os mandados. Campos como agente, arma e faixa etária vêm vazios.

In [ ]:
if dados_brutos_2026 is not None:
    dados_brutos_2026.info()
    display(dados_brutos_2026.head())
else:
    print("Inspeção bruta disponível apenas com o XLSX oficial (ver scripts/etl_sinesp.py, etapas 04 a 06).")

## 6. Tratamento

No ETL, cada linha original vira uma observação padronizada: `ano`, `mes`, `evento`, `abrangencia`, `unidade` e `valor`.
- A data vem do serial do Excel (dias desde 30/12/1899) e precisa cair no dia 1 do mês.
- Vítimas vêm de `total_vitima`; mandados, de `total` — unidades diferentes, **nunca somadas**.
- O valor é inteiro não negativo; quando a fonte não informa, fica ausente (`<NA>`), nunca zero.

Aqui conferimos o resultado na base aprovada: tipos das colunas e as unidades usadas.

In [ ]:
print(dados_sinesp.dtypes)
print()
# value_counts conta quantas observações há de cada unidade.
print(dados_sinesp["unidade"].value_counts())

## 7. Verificação de duplicidades

Cada série (evento × abrangência) deve ter **uma** observação por mês. `duplicated()` marca repetições da chave. Nenhuma linha é apagada: se houvesse repetição, ela seria investigada (o Sinesp só admite parcelas somadas em *Tentativa de homicídio* e *Estupro*, conforme a nota oficial).

In [ ]:
chave_observacao = ["evento", "abrangencia", "ano", "mes"]
repetidas = dados_sinesp.duplicated(subset=chave_observacao, keep=False)
print(f"Observações com chave repetida: {int(repetidas.sum())}")

## 8. Valores ausentes e zeros

- **Zero publicado**: a fonte informou 0 no mês — é uma observação válida.
- **Valor ausente**: a fonte não informou — nunca é trocado por zero.

`isna()` marca ausentes e `eq(0)` marca zeros.

In [ ]:
print(f"Valores ausentes: {int(dados_sinesp['valor'].isna().sum())}")
print(f"Zeros publicados: {int(dados_sinesp['valor'].eq(0).sum())}")

# Zeros por série: séries de eventos raros (latrocínio, feminicídio) têm muitos meses com zero.
zeros_por_serie = dados_sinesp.assign(zero=dados_sinesp["valor"].eq(0)).groupby(["evento", "abrangencia"])["zero"].sum()
zeros_por_serie.sort_values(ascending=False).to_frame("meses com zero")

## 9. Recorte Joinville e cobertura das séries

Toda a base é de Joinville/SC. O catálogo mostra, por série, o primeiro e o último mês publicados. Repare que a PRF termina em agosto/2026 e os mandados da PF começam em janeiro/2025.

In [ ]:
catalogo = dp.carregar_catalogo_sinesp()
catalogo[["evento", "abrangencia", "unidade", "inicio", "fim", "meses_observados", "meses_valor_zero"]]

## 10. Estatísticas

Usamos como exemplo **homicídio doloso (Estadual)**. As estatísticas descrevem os valores **mensais**:
- soma, média (`mean`), mediana (`median`), mínimo e máximo;
- desvio-padrão com `ddof=0` (descreve os meses observados, sem inferência);
- quartis (`quantile`), IQR e limites de Tukey para identificar meses fora do padrão.

In [ ]:
INDICADOR = "Homicídio doloso"
ABRANGENCIA = "Estadual"
ANOS = list(range(2020, 2027))

serie = calculos.serie_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA)
resumo = calculos.resumo_serie_sinesp(serie)
pd.Series(resumo).to_frame("valor")

In [ ]:
# Totais por ano: só os anos completos (12 meses) recebem total anual; 2026 é parcial.
tabela_anual = calculos.tabela_anual_sinesp(serie)
tabela_anual[["ano", "cobertura", "meses_presentes", "soma"]]

In [ ]:
# Períodos equivalentes: janeiro até o último mês publicado (setembro), em todos os anos.
periodos = calculos.comparacao_periodo_sinesp(serie, calculos.mes_final_da_serie(serie))
periodos[["ano", "periodo", "soma"]]

In [ ]:
# Meses fora dos limites de Tukey (Q1 − 1,5·IQR; Q3 + 1,5·IQR). O diagnóstico diz se o método é informativo para a série.
diagnostico = calculos.diagnostico_outliers_iqr(serie)
print(diagnostico["rotulo"], "—", diagnostico["mensagem"])
diagnostico["fora"]

## 11. Gráficos e interpretações

As figuras são as mesmas do painel: barras para a evolução anual e mensal, perfil mensal pela mediana (só anos completos) e a série mensal com os limites de Tukey.

In [ ]:
graficos.figura_serie_anual_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA, ANOS)

**Interpretação:** o total publicado caiu de 62 vítimas em 2020 para 29 em 2025, com oscilações (49 em 2022 e 55 em 2023). É uma descrição das contagens publicadas, não uma explicação de causas.

In [ ]:
graficos.figura_mensal_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA, [2025, 2026])

**Interpretação:** em janeiro–setembro foram 21 vítimas em 2025 e 25 em 2026. Contagens mensais pequenas variam bastante de um mês para outro; a comparação correta é entre períodos de mesma duração.

In [ ]:
graficos.figura_sazonalidade_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA, ANOS)

**Interpretação:** o perfil usa só os seis anos completos (2020–2025). Com tão poucos anos, ele descreve esses anos e não demonstra sazonalidade.

In [ ]:
graficos.figura_outliers_iqr_sinesp(dados_sinesp, INDICADOR, ABRANGENCIA, ANOS)

**Interpretação:** janeiro/2020 (10) e janeiro/2023 (19) ficam acima do limite superior (9,5). São meses atípicos em relação ao conjunto observado, não prova de erro ou de anomalia.

## 12. Conclusões

- Homicídio doloso em Joinville caiu nos anos completos de 2020 a 2025; em 2026 (janeiro–setembro) o total ficou acima do mesmo período de 2025.
- As séries são contagens mensais pequenas: diferenças absolutas são mais informativas que percentuais.
- O critério IQR só é informativo em 5 das 12 séries; nas demais (muitos zeros ou contagens muito baixas) nenhum mês é apontado como atípico.

In [ ]:
diagnosticos = calculos.diagnostico_outliers_todas_as_series(dados_sinesp)
diagnosticos[["evento", "abrangencia", "meses", "zeros", "classificacao"]]

## 13. Limitações

- Registros administrativos agregados: não medem toda a criminalidade, não estimam subnotificação e não indicam causa.
- Sem taxas populacionais, dados por bairro, horário ou corporação; sem total geral entre indicadores ou abrangências.
- 2026 é parcial e a PRF termina antes das demais séries.
- Furto, roubo, estupro e tráfico **não existem** no Sinesp de Joinville; publicações municipais divergem e não entram na base (ver [README, fontes complementares](../README.md#44-fontes-complementares-investigação-preservada)).

## 14. Exploração separada: violência doméstica SSP/SC

### Fonte, recorte e conferência

Três relatórios oficiais — 1º e 2º semestres de 2023 e ano de 2024 — em `dados/brutos/ssp_sc/pdfs/`, com CSVs posicionais em `csv_extraidos/`.
O módulo [ssp_sc.py](../src/pipeline/ssp_sc.py) lê ano e meses nos PDFs, identifica o bloco JOINVILLE, pareia as páginas e confere as células contra as tabelas oficiais.
As grafias “0 Dolosa” e “- Dolosa” são normalizadas apenas no nome; os rótulos originais e valores são preservados.

**Unidade em validação:** “Fato Comunicado” não possui definição operacional nos PDFs. Não interpretar como pessoas, vítimas ou ocorrências únicas,
nem somar categorias ou comparar seus totais com o Sinesp. Esta fonte não integra o Streamlit.
Sem o acervo completo, esta seção informa explicitamente que não foi executada; as demais análises continuam disponíveis.

In [ ]:
from src.pipeline import ssp_sc

resultado_ssp = None
if ssp_sc.acervo_disponivel():
    resultado_ssp = ssp_sc.processar()  # em memória: não altera nenhum dado
    vd = resultado_ssp["vd_extraida"]
    assert len(vd) == 216 and not vd.duplicated(["ano", "mes", "categoria"]).any()
    assert vd["fatos_comunicados"].eq(0).sum() == 27
    assert resultado_ssp["verificacoes"]["ok"].all()
    print("SSP/SC: 216 células; 9 categorias; 24 meses; 27 zeros; unidade em validação.")
    print("Conferências aprovadas:", resultado_ssp["resumo"]["verificacoes_aprovadas"])
    display(resultado_ssp["rastreabilidade"].head())
else:
    print("SSP/SC NÃO EXECUTADA: requer os 45 PDFs e 45 CSVs do acervo local. Nenhum valor foi estimado.")

### Estatísticas por categoria

Somamos os 12 meses dentro de cada categoria e ano; conferimos que cada combinação tem exatamente 12 valores.
A média e o desvio-padrão (`ddof=0`) descrevem os 24 meses observados. Não são estimativas de risco nem intervalos de confiança.

In [ ]:
if resultado_ssp is not None:
    grupos_vd = vd.groupby(["categoria", "ano"])["fatos_comunicados"]
    assert grupos_vd.count().eq(12).all()
    totais_vd = grupos_vd.sum(min_count=12).unstack("ano")
    totais_vd["diferença 2024 − 2023"] = totais_vd[2024] - totais_vd[2023]
    display(totais_vd.sort_values(2024, ascending=False))
    estatisticas_vd = vd.groupby("categoria")["fatos_comunicados"].agg(
        meses="count", media="mean", mediana="median", minimo="min", maximo="max", dp=lambda x: x.std(ddof=0))
    display(estatisticas_vd.round(2))

### Comparação de anos completos

O gráfico compara cada categoria consigo mesma. A imagem exportada pelo módulo fica em
[`documentacao/graficos/painel/07-sspsc_violencia_domestica_2023_2024.png`](../documentacao/graficos/painel/07-sspsc_violencia_domestica_2023_2024.png).
Na mesma pasta, os arquivos 01 a 06 são capturas da interface; o arquivo 07 é esta figura exploratória SSP/SC, que não integra o dashboard.

In [ ]:
if resultado_ssp is not None:
    display(ssp_sc.figura_violencia_domestica(vd))

**Leitura exploratória:** ameaça passa de 1.957 para 2.085 fatos comunicados; injúria, de 541 para 716;
lesão corporal leve, de 989 para 959. Sete categorias têm mais registros em 2024 e duas têm menos.
Isso descreve registros publicados; não demonstra variação do número de vítimas nem explica causas.

### Rastreabilidade e republicações

As 216 células de violência doméstica vêm de um único relatório por mês. Já os boletins de homicídio publicam 294 valores:
58 grupos de republicação (17 concordantes, 37 inconsistências sem explicação e 4 possíveis problemas de cabeçalho).
Nenhuma versão é escolhida. Nos 30 valores sem republicação, os campos de grupo ficam ausentes, nunca convertidos em zero ou “não divergente”.

In [ ]:
if resultado_ssp is not None:
    display(resultado_ssp["grupos"].groupby("classificacao").size().to_frame("grupos"))
    sem_grupo = resultado_ssp["versoes"]["grupo_republicacao"].isna()
    assert sem_grupo.sum() == 30
    assert resultado_ssp["versoes"].loc[sem_grupo, "grupo_divergente"].isna().all()
    assert not (RAIZ / "dados/processados/joinville_indicadores_consolidados.csv").exists()

## 15. Síntese e próximos passos

O Sinesp sustenta comparações descritivas por indicador, abrangência e períodos equivalentes. A SSP/SC acrescenta uma exploração separada,
cuja extração é rastreável, mas cuja unidade ainda precisa de confirmação oficial. A validação técnica da extração não resolve a definição metodológica.

Antes de incluir SSP/SC no painel: esclarecer “Fato Comunicado”, resolver versões de homicídio e confirmar condições de reutilização.
Não calcular soma de fontes, total geral de criminalidade, taxas sem denominador ou conclusões causais.
